# Live compiled semantic predicates inside HNSW

This is the decisive follow-up to the precomputed semantic-HNSW experiment.

**Frozen search algorithm:** dense HNSW navigation; the semantic predicate only decides result eligibility; invalid nodes remain traversable; no semantic steering and no 2-hop bridge expansion.

The new `semantic_hnsw_live` method executes the real **Binary1-LS2-int4** programs from the 56-byte item representation *inside the timed graph traversal*. Semantic values are cached once per node per query. A dense admissibility check happens before semantic execution, so nodes that cannot improve the valid beam are rejected without touching the semantic program.

We compare against three materialized lower-bound baselines: post-filter HNSW, library filtered HNSW, and the same custom traversal with precomputed semantic scores. Live quality should match the custom materialized method exactly; the latency gap is the cost of executing the compiled programs.


In [ ]:
#@title 1) Settings
FULL_DATA = False #@param {type:"boolean"}
QUERIES = 100 #@param {type:"integer"}
K = 50 #@param {type:"integer"}
EF = 128 #@param {type:"integer"}
M = 24 #@param {type:"integer"}
EF_CONSTRUCTION = 200 #@param {type:"integer"}
POSTFILTER_OVERSAMPLE = 8 #@param {type:"integer"}
POSITIVE = 'minimalist,office_appropriate' #@param {type:"string"}
NEGATIVE = 'technical_sporty' #@param {type:"string"}
TARGET_FRACTIONS = '0.50,0.20,0.10,0.05,0.02,0.01' #@param {type:"string"}
print({'FULL_DATA':FULL_DATA,'QUERIES':QUERIES,'K':K,'EF':EF,'POSITIVE':POSITIVE,'NEGATIVE':NEGATIVE,'TARGET_FRACTIONS':TARGET_FRACTIONS})


In [ ]:
#@title 2) Clone repo + dependencies
import os, pathlib, shutil, subprocess, sys
ROOT=pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','faiss-cpu'],check=True)
if shutil.which('rustc') is None or shutil.which('cargo') is None:
    print('Installing minimal stable Rust...')
    subprocess.run(['bash','-lc',"curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"],check=True)
    os.environ['PATH']=str(pathlib.Path.home()/'.cargo'/'bin')+os.pathsep+os.environ.get('PATH','')
print('commit:',subprocess.check_output(['git','rev-parse','HEAD']).decode().strip())
print('python:',sys.version.split()[0])
print('rustc:',subprocess.check_output(['rustc','--version']).decode().strip())


In [ ]:
#@title 3) Export fashion embeddings + compiled Binary1-LS2-int4 programs
import pathlib, shutil, subprocess, sys, time, os
os.chdir('/content/ras')
CFG='configs/binary_bbq.yaml' if FULL_DATA else 'configs/binary_bbq_smoke.yaml'
ASSETS=pathlib.Path('/content/semantic_hnsw_live_assets')
if ASSETS.exists(): shutil.rmtree(ASSETS)
t0=time.time()
subprocess.run([sys.executable,'-m','experiments.export_native_finalists','--config',CFG,'--out-dir',str(ASSETS)],check=True)
print(f'export finished in {(time.time()-t0)/60:.1f} min')
print('programs:',sorted(p.name for p in (ASSETS/'sidecar_programs').iterdir() if p.is_dir()))


In [ ]:
#@title 4) Compile live semantic-HNSW Rust executable
import subprocess, os, time
os.chdir('/content/ras')
t0=time.time()
subprocess.run(['cargo','build','--release','--manifest-path','rust/semantic_engine/Cargo.toml','--bin','semantic_hnsw_live'],check=True)
BIN='/content/ras/rust/semantic_engine/target/release/semantic_hnsw_live'
print(f'compiled in {time.time()-t0:.1f}s:',BIN)


In [ ]:
#@title 5) Convert target selectivities into semantic gates
import numpy as np, pandas as pd, sys, pathlib, importlib
SRC=str(pathlib.Path('/content/ras/src'))
sys.path[:]=[p for p in sys.path if p != SRC]
sys.path.insert(0,SRC)
for name in list(sys.modules):
    if name=='ras' or name.startswith('ras.'): del sys.modules[name]
importlib.invalidate_caches()
from ras import SemanticExecutor
pos=[x.strip() for x in POSITIVE.split(',') if x.strip()]
neg=[x.strip() for x in NEGATIVE.split(',') if x.strip()]
n_pred=len(pos)+len(neg)
executor=SemanticExecutor.open(str(ASSETS/'sidecar_index'),str(ASSETS/'sidecar_programs'))
n_items=executor.index.n_items
ids=np.arange(n_items,dtype=np.int64)
sem_mean=executor.score_candidates(ids,positive=pos,negative=neg)/max(1,n_pred)
rows=[]
for f in [float(x.strip()) for x in TARGET_FRACTIONS.split(',') if x.strip()]:
    if not (0<f<=1): continue
    if f*n_items < K+1:
        print(f'skipping {f:.3f}: only ~{f*n_items:.1f} eligible items for K={K}')
        continue
    gate=float(np.quantile(sem_mean,1.0-f))
    actual=float(np.mean(sem_mean>=gate))
    rows.append({'target_fraction':f,'gate_logprob':gate,'actual_fraction_python':actual,'eligible_items':int((sem_mean>=gate).sum())})
gate_df=pd.DataFrame(rows).sort_values('target_fraction',ascending=False).reset_index(drop=True)
display(gate_df)
assert len(gate_df)


In [ ]:
#@title 6) Run LIVE semantic execution across selectivities
import subprocess, pathlib, time, pandas as pd
all_runs=[]
for r in gate_df.itertuples(index=False):
    frac=float(r.target_fraction); gate=float(r.gate_logprob)
    out=pathlib.Path(f'/content/semantic_hnsw_live_{frac:.3f}.csv')
    cmd=[BIN,'--assets',str(ASSETS),'--programs',str(ASSETS/'sidecar_programs'),'--positive',POSITIVE,'--negative',NEGATIVE,'--queries',str(QUERIES),'--k',str(K),'--ef',str(EF),'--m',str(M),'--ef-construction',str(EF_CONSTRUCTION),'--gate-logprob',str(gate),'--postfilter-oversample',str(POSTFILTER_OVERSAMPLE),'--out',str(out)]
    print(f'\n=== target {frac:.3f}, gate {gate:.5f} ===')
    t0=time.time(); run=subprocess.run(cmd,text=True,capture_output=True)
    print(run.stdout)
    if run.returncode!=0:
        print(run.stderr); raise RuntimeError(f'run failed for {frac}: {run.returncode}')
    z=pd.read_csv(out); z['target_fraction']=frac; z['gate_logprob']=gate; all_runs.append(z)
    print(f'wall time: {time.time()-t0:.2f}s')
results=pd.concat(all_runs,ignore_index=True)
print('rows:',len(results))


In [ ]:
#@title 7) Live latency / recall / predicate-work summary
import numpy as np, pandas as pd
summary=(results.groupby(['target_fraction','method']).agg(queries=('query_id','count'),mean_latency_ms=('latency_ms','mean'),p50_latency_ms=('latency_ms','median'),p95_latency_ms=('latency_ms',lambda x: np.quantile(x,.95)),mean_recall_at_k=('recall_at_k','mean'),mean_returned=('returned','mean'),mean_visited=('visited','mean'),mean_semantic_evals=('semantic_evals','mean'),mean_predicate_evals=('predicate_evals','mean'),mean_dense_pruned_before_semantic=('dense_pruned_before_semantic','mean'),qualified_fraction=('qualified_fraction','mean'),live_match_rate=('live_matches_materialized','mean')).reset_index())
display(summary.sort_values(['target_fraction','mean_latency_ms'],ascending=[False,True]))

# Direct decomposition: how much latency do the real programs add over the same traversal with free semantic lookup?
piv_t=summary.pivot(index='target_fraction',columns='method',values='mean_latency_ms')
piv_r=summary.pivot(index='target_fraction',columns='method',values='mean_recall_at_k')
piv_e=summary.pivot(index='target_fraction',columns='method',values='mean_predicate_evals')
rows=[]
for f in sorted(piv_t.index,reverse=True):
    row={'target_fraction':f}
    if 'semantic_hnsw_live' in piv_t.columns and 'custom_hnsw_materialized' in piv_t.columns:
        overhead=float(piv_t.loc[f,'semantic_hnsw_live']-piv_t.loc[f,'custom_hnsw_materialized'])
        pe=float(piv_e.loc[f,'semantic_hnsw_live'])
        row['live_latency_ms']=float(piv_t.loc[f,'semantic_hnsw_live'])
        row['materialized_custom_ms']=float(piv_t.loc[f,'custom_hnsw_materialized'])
        row['live_program_overhead_ms']=overhead
        row['live_recall']=float(piv_r.loc[f,'semantic_hnsw_live'])
        row['predicate_evals']=pe
        row['approx_ns_per_predicate_eval']=overhead*1e6/pe if pe>0 else np.nan
    if 'hnsw_filtered_materialized' in piv_t.columns:
        row['live_over_filtered_materialized_latency']=float(piv_t.loc[f,'semantic_hnsw_live']/piv_t.loc[f,'hnsw_filtered_materialized'])
    rows.append(row)
comparison=pd.DataFrame(rows)
display(comparison)

live=results[results.method=='semantic_hnsw_live']
print('live/materialized exact-ID parity:',float(live.live_matches_materialized.mean()))
assert live.live_matches_materialized.all(), 'Live execution changed result IDs; inspect before interpreting latency.'


In [ ]:
#@title 8) Plot live vs materialized recall-latency frontier
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,6))
for method,g in summary.groupby('method'):
    g=g.sort_values('target_fraction',ascending=False)
    ax.plot(g.mean_latency_ms,g.mean_recall_at_k,marker='o',label=method)
    for _,r in g.iterrows(): ax.annotate(f'{int(round(100*r.target_fraction))}%',(r.mean_latency_ms,r.mean_recall_at_k),xytext=(4,4),textcoords='offset points',fontsize=8)
ax.set_xlabel('Mean query latency (ms)')
ax.set_ylabel(f'Mean Recall@{K}')
ax.set_title('Live compiled semantic predicates inside HNSW')
ax.grid(True,alpha=.25); ax.legend(); plt.show()


In [ ]:
#@title 9) Package results
import pathlib, shutil, json, platform, subprocess
PKG=pathlib.Path('/content/semantic_hnsw_live_artifact')
if PKG.exists(): shutil.rmtree(PKG)
PKG.mkdir()
results.to_csv(PKG/'per_query.csv',index=False); summary.to_csv(PKG/'summary.csv',index=False); comparison.to_csv(PKG/'comparison.csv',index=False)
meta={'commit':subprocess.check_output(['git','-C','/content/ras','rev-parse','HEAD']).decode().strip(),'full_data':FULL_DATA,'queries':QUERIES,'k':K,'ef':EF,'m':M,'positive':POSITIVE,'negative':NEGATIVE,'timing_scope':'semantic_hnsw_live includes real Binary1-LS2-int4 program execution inside timed HNSW traversal; materialized baselines use precomputed semantic scores','cpu':pathlib.Path('/proc/cpuinfo').read_text().split('model name')[1].split('\n')[0].split(':',1)[-1].strip() if pathlib.Path('/proc/cpuinfo').exists() and 'model name' in pathlib.Path('/proc/cpuinfo').read_text() else 'unknown'}
(PKG/'environment.json').write_text(json.dumps(meta,indent=2)); shutil.make_archive('/content/rsa_semantic_hnsw_live','zip',PKG)
print('/content/rsa_semantic_hnsw_live.zip')
